## **LangChain**

 <a href="#LangChain-concepts">LangChain concepts</a>
        <ol>
            <li><a href="#Model">Model</a></li>
            <li><a href="#Chat-model">Chat model</a></li>
            <li><a href="#Chat-message">Chat message</a></li>
            <li><a href="#Prompt-templates">Prompt templates</a></li>
            <li><a href="#Example-selectors">Example selectors</a></li>
            <li><a href="#Output-parsers">Output parsers</a></li>
            <li><a href="#Documents">Documents</a></li>
            <li><a href="#Memory">Memory</a></li>
            <li><a href="#Chains">Chains</a></li>
            <li><a href="#Agents">Agents</a></li>
        </ol>

In [45]:
import dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import  PromptTemplate, ChatPromptTemplate
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.example_selectors import LengthBasedExampleSelector
from langchain_core.prompts import FewShotPromptTemplate
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama 
from langchain_classic.memory import ChatMessageHistory
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain, LLMChain, SequentialChain

### 1 & 2. Model and Chat Model 

In [2]:
Model_API = dotenv.get_key("../.env", "GEMINI_API")

In [3]:
def get_chat_llm (params=None):
    
    # llm = ChatGoogleGenerativeAI(
    #     model = "gemini-2.5-flash",
    #     temperature=0.3,
    #     api_key=Model_API
    # )
    
    llm = Ollama(model="llama3.1")
    
    return llm

In [4]:
llm_model = get_chat_llm()

C:\Users\Vish\AppData\Local\Temp\ipykernel_26096\3330961671.py:9: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1")


### 3. Chat Message 

In [7]:
message = llm_model.invoke([
    SystemMessage("You are a helpful assistant to find best car based on the user requirement in a one short sentence"),
    HumanMessage("I am more interested about sport sedan. please suggest me a one")
])

In [8]:
print(message)

Based on your preference for a sport sedan, I would recommend the BMW M3 as it offers excellent performance, handling, and style with its powerful engine and agile suspension.


### 4. Prompt templates

We use prompt templates to give single or multi messages to the model to get more clear and accurate reponse for our questions.

##### Promt Template

**Prompt Template** uses to give single string input for the model as a message 

In [7]:
prompt = PromptTemplate.from_template("Tell me a {type} about the {topic}")
input_ = {"type":"joke", "topic":"computer"}

prompt.invoke(input_)

StringPromptValue(text='Tell me a joke about the computer')

In [8]:
output = prompt | llm_model
output

PromptTemplate(input_variables=['topic', 'type'], input_types={}, partial_variables={}, template='Tell me a {type} about the {topic}')
| ChatGoogleGenerativeAI(profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', temperature=0.3, client=<google.genai.client.Client object at 0x000001F46FED8150>, default_metadata=(), model_kwargs={})

##### Chat Prompt Template

These prompt templates are used to format a list of messages. These "templates" consist of a list of templates themselves.


In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "tell me a joke about {topic}")
])

input_ = {"topic":"cat"}

prompt.invoke(input_)

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='tell me a joke about cat', additional_kwargs={}, response_metadata={})])

#### Message PlaceHolder

Message Placeholder, we use to apply relevant list of messages to a paticular positions in the chat templates.

In [10]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant to user"),
    MessagesPlaceholder("msgs")
])

input_ = {
    "msgs": [
        HumanMessage(content='What is the world largest mountain?'),
        HumanMessage(content="Please give me a short description about toyota supra")
    ]
}

formatted_prompt = chat_prompt.invoke(input_)
formatted_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant to user', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the world largest mountain?', additional_kwargs={}, response_metadata={}), HumanMessage(content='Please give me a short description about toyota supra', additional_kwargs={}, response_metadata={})])

In [11]:
response = llm_model.invoke(formatted_prompt)
response

AIMessage(content="The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China (Tibet).\n\n---\n\nThe **Toyota Supra** is an iconic Japanese sports car and grand tourer produced by Toyota. It's renowned for its powerful **inline-six engines**, particularly the legendary **2JZ-GTE** found in the fourth-generation (Mk4) model, which made it a favorite for tuning and high-performance modifications. The Supra gained significant pop culture fame, notably from the *Fast & Furious* film franchise. After a hiatus, it was revived as the fifth-generation (A90/Mk5) in collaboration with BMW, continuing its legacy as a performance-oriented coupe.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider':

In [12]:
print(response.content)

The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**. It stands at **8,848.86 meters (29,031.7 feet)** above sea level and is located in the Mahalangur Himal sub-range of the Himalayas, straddling the border between Nepal and China (Tibet).

---

The **Toyota Supra** is an iconic Japanese sports car and grand tourer produced by Toyota. It's renowned for its powerful **inline-six engines**, particularly the legendary **2JZ-GTE** found in the fourth-generation (Mk4) model, which made it a favorite for tuning and high-performance modifications. The Supra gained significant pop culture fame, notably from the *Fast & Furious* film franchise. After a hiatus, it was revived as the fifth-generation (A90/Mk5) in collaboration with BMW, continuing its legacy as a performance-oriented coupe.


In [13]:
chain = chat_prompt | llm_model
response = chain.invoke(input_)
print(response.content)

The world's largest mountain, in terms of **highest peak above sea level**, is **Mount Everest**.

---

The **Toyota Supra** is an iconic Japanese sports car known for its performance, rear-wheel drive layout, and powerful inline-six engines. The **fourth-generation (Mk4)**, produced from 1993-2002, is particularly legendary, largely due to its highly tunable 2JZ-GTE engine and appearances in popular culture like the *Fast & Furious* film series. After a long hiatus, the Supra returned in 2019 (the **fifth-generation or A90/A91**) as a collaboration with BMW, maintaining its focus on performance and driver engagement.


### 5. Example Selector

If you want to provide your model with examples and you have a large number of them, you need to filter out and select the most relevant examples for a given query. This task is performed by **Example Selectors**.

There are main examples selectors:

* `Similarity`: Uses semantic similarity between inputs and examples to decide which examples to choose.
* `MMR`: Uses Max Marginal Relevance between inputs and examples to decide which examples to choose.
* `Length`: Selects examples based on how many can fit within a certain length
* `Ngram`: Uses ngram overlap between inputs and examples to decide which examples to choo

In [14]:
examples = [
    {"input": "happy", "output": "sad"},
    {"input": "tall", "output": "short"},
    {"input": "energetic", "output": "lethargic"},
    {"input": "sunny", "output": "gloomy"},
    {"input":"windy", "output":"calm"}
]
example_prompt = PromptTemplate(
    template= "input: {input}\noutput: {output}",
    input_variables=['input', 'output']
)
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=25
)

## for example driven template, we use fewshot prompt template
dynamic_prompt = FewShotPromptTemplate(
    example_selector = example_selector,
    example_prompt = example_prompt,
    prefix = "Give the antonym of every input",
    suffix = "input: {input}\noutput:",
    input_variables = ['input']
)

In [15]:
print(dynamic_prompt.format(input = "big"))

Give the antonym of every input

input: happy
output: sad

input: tall
output: short

input: energetic
output: lethargic

input: sunny
output: gloomy

input: windy
output: calm

input: big
output:


In [16]:
long_string = "big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else"
print(dynamic_prompt.format(input = long_string))

Give the antonym of every input

input: happy
output: sad

input: big and huge and massive and large and gigantic and tall and much much much much much bigger than everything else
output:


### 6. Output Parsers

Output parsers are used to convert the raw responses generated by an LLM into a more structured and useful format. They are especially helpful when working with structured data or when standardizing outputs from different language and chat models.

LangChain provides a variety of output parsers to handle different formatting needs. In this lab, you will work with the following two parsers:

* JSON Parser: Produces output in JSON format according to a specified schema. It can also utilize a Pydantic model to generate structured JSON data, making it one of the most dependable options for obtaining structured output without relying on function calling.

* CSV Parser: Formats the model's output as a list of comma-separated values (CSV), which is useful for tabular data representation.

In [17]:
class Car(BaseModel):
    query:str = Field(description="question to setup a car")
    response:str = Field(description="answer to resolve the car")

In [18]:
car_query = "tell me the information about Lambogini"

output_parser = JsonOutputParser(pydantic_object=Car)
format_instruction = output_parser.get_format_instructions()

prompt  = PromptTemplate(
    template= "Answer the user query.\n {format_instruction}\n{query}",
    input_variables=['query'],
    partial_variables= {"format_instruction": format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":car_query })
print(response)


{'query': 'tell me the information about Lambogini', 'response': "Lamborghini (officially Automobili Lamborghini S.p.A.) is an Italian brand and manufacturer of luxury sports cars and SUVs based in Sant'Agata Bolognese. The company is owned by the Volkswagen Group through its subsidiary Audi. Founded by Ferruccio Lamborghini in 1963, the company was created with the aim of competing with established marques like Ferrari. Lamborghini is renowned for its high-performance vehicles, often characterized by their distinctive, aggressive styling and powerful engines, typically V10 or V12. Iconic models include the Miura, Countach, Diablo, Murciélago, Gallardo, Aventador, Huracán, and the Urus SUV."}


In [19]:
user_query = "Give me details about DPO fine-tune"

output_parser = CommaSeparatedListOutputParser()
format_instruction = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="Answer the user query\n{format_instruction}\n{query}",
    input_variables=["query"],
    partial_variables={"format_instruction":format_instruction}
)

chain = prompt | llm_model | output_parser

response = chain.invoke({"query":user_query})
print(response)

['Direct Preference Optimization', 'Aligns LLMs with human preferences', 'Does not require a separate reward model', 'Uses preference data (chosen vs. rejected responses)', 'Simpler and more stable than PPO-based RLHF', 'Optimizes a direct analytical loss function', 'Based on a supervised fine-tuned (SFT) model', 'Computationally efficient', 'Achieves comparable or superior performance to RLHF', 'Avoids complexities of reinforcement learning']


## **LangChain Components for RAG Application**

### 7. Documents

#### Document Object

The document object contains two main components:

1. page-content: this represents the content of the whole document.
2. metadata: this represents the all the attributes that paticularlly unique for the each document. this availables at Dict types

examples:
document_id, timestamp, document_title, author, etc

In [20]:
document = Document(
    page_content="This is a example for document attribute which is page_content",
    metadata={
        "document_id": 1294857,
        "document_source": "about firewall",
        "created_time": 21344656
    }
)

document.page_content

'This is a example for document attribute which is page_content'

#### Document Loaders

Document loaders in LangChain are used to import data from multiple sources. For example, you can load a PDF file and make it readable for an LLM using LangChain.

LangChain provides more than 100 different document loaders, along with integrations with major platforms like AirByte and Unstructured. These integrations allow you to load various types of content such as HTML, PDFs, and source code from different locations, including private S3 buckets and public websites.

A full list of supported document types can be found here: https://python.langchain.com/v0.1/docs/integrations/document_loaders/

In [12]:
file_path = "https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf"
loader = PyPDFLoader(file_path)

In [13]:
loader

In [14]:
document = loader.load()
document[0]

Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'author': '', 'keywords': '', 'moddate': '2024-11-06T10:08:55+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='LangChain\nVasilios Mavroudis\nAlan Turing Institute\nvmavroudis@turing.ac.uk\nAbstract. LangChain is a rapidly emerging framework that offers a ver-\nsatile and modular approach to developing applications powered by large\nlanguage models (LLMs). By leveraging LangChain, developers can sim-\nplify complex stages of the application lifecycle—such as development,\nproductionization, and deployment—making it easier to build scalable,\nstateful, and contextually aware applications. It provides tool

#### URL and Website Loaders

In [9]:
web_loader = WebBaseLoader("https://python.langchain.com/v0.2/docs/introduction/")
web_data = web_loader.load()

In [10]:
web_data[0].page_content[:1000]

'LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageOpen sourceSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewDeep AgentsLangChainLangGraphIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatternsIntegrationsAdvanced usageGuardrailsRuntimeContext engineeringModel Context Protocol (MCP)Human-in-the-loopMulti-agentRetrievalLong-term memoryAgent developmentLangSmith StudioTestAgent Chat UIDeploy with LangSmithDeploymentObservabilityOn this page Create an agent Core benefitsLangChain overviewCopy pageLangChain provides create_agent: a minimal, highly configurable age

#### Text-Splitter

After loading documents, you often need to transform them so they work better for your specific use case.

A common step is breaking a long document into smaller chunks that can fit within a model’s context window. LangChain provides several built-in document transformers that help with splitting, merging, filtering, and other ways of processing documents.

In general, text splitters work in the following way:

1. The text is first divided into small, meaningful units, such as sentences.
2. These small parts are then gradually combined into larger chunks until a defined size limit is reached (based on a chosen measurement method).
3. Once a chunk reaches the limit, it is stored as a separate segment, and a new chunk begins. Often, a small overlap is kept between chunks to preserve context across sections.

You can view the different types of text splitters supported by LangChain here:
[https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/)

As an example, we will use the simple `CharacterTextSplitter` to divide the LangChain paper you previously loaded. This basic splitter works by breaking text using characters (defaulting to `"\n\n"`) and measuring chunk size based on the number of characters.


In [15]:
text_splitter = CharacterTextSplitter(chunk_size = 700, chunk_overlap=20, separator="\n")
chunks = text_splitter.split_documents(document)
print(len(chunks))

60


In [16]:
print(chunks[8].page_content)

deployment of LangChain applications. Finally, section 5 addresses the limita-
tions and criticisms of LangChain, particularly focusing on the complexities and
security concerns associated with its modular design and external integrations.
1 Architecture
LangChain is built with a modular architecture, designed to simplify the life-
cycle of applications powered by large language models (LLMs), from initial
development through to deployment and monitoring [3]. This modularity al-
lows developers to configure, extend, and deploy applications tailored to specific


#### Embedding Models

Embedding models are designed to work directly with text embeddings.

They convert a piece of text into a numerical vector representation. This is useful because it lets you represent text in a vector space, where similar meanings are positioned closer together. As a result, you can perform tasks like semantic search by finding text segments that are closest to each other in that vector space.


In [5]:
embeddings = HuggingFaceEmbeddings(
    model="BAAI/bge-m3",
    
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [6]:
text = "Hi im seran"
embeds = embeddings.embed_query("Hi im seran")
embeds

[-0.03832868114113808,
 0.018502652645111084,
 -0.04393254965543747,
 -0.0003660297370515764,
 -0.011765962466597557,
 -0.01035870797932148,
 -0.01746307872235775,
 0.005218490958213806,
 0.003107429714873433,
 0.015090255066752434,
 0.01508807111531496,
 0.009850089438259602,
 0.000960557081270963,
 -0.001512787421233952,
 -0.023006483912467957,
 -0.009658196941018105,
 0.03315352648496628,
 -0.02878696098923683,
 0.04440535604953766,
 0.022929757833480835,
 -0.0315418504178524,
 0.00619888911023736,
 -0.008088978007435799,
 0.004414239898324013,
 0.03774955868721008,
 0.052773598581552505,
 -0.015159388072788715,
 0.04066237062215805,
 -0.024652661755681038,
 -0.009521335363388062,
 -0.00772809935733676,
 0.02571774832904339,
 -0.0014926429139450192,
 -0.0699140727519989,
 -0.0077828457579016685,
 -0.0338699072599411,
 -0.02853497490286827,
 -0.034026388078927994,
 -0.036711424589157104,
 0.06146254390478134,
 -0.006368113681674004,
 -0.008128105662763119,
 0.01568002440035343,
 -0.0

In [17]:
## to embedding documents
text = [doc.page_content for doc in chunks]
document_embeds = embeddings.embed_documents(text[0:100])

In [18]:
print(len(document_embeds))

60


#### Vector Stores

One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A [vector store](https://python.langchain.com/v0.1/docs/modules/data_connection/vectorstores/) takes care of storing embedded data and performing vector search for you.


We use **Chroma** here to demonstrase the vector stores

In [19]:
# 1st initiate chroma object using document chunks and embeddings
doc_search = Chroma.from_documents(chunks, embeddings)

query = "Langchain"

#if we want to find the relevant context for query
docs = doc_search.similarity_search(query)
docs

[Document(id='6eb1e3f7-d40c-481f-a04b-c730a590a786', metadata={'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'page_label': '11', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.26', 'keywords': '', 'trapped': '/False', 'total_pages': 14, 'author': '', 'subject': '', 'moddate': '2024-11-06T10:08:55+00:00', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'page': 10, 'title': '', 'creationdate': '2024-11-06T10:08:55+00:00'}, page_content='leverage LangChain’s extensive library of tools, models, and connectors as part'),
 Document(id='7fe26993-e993-46c8-906b-182641f1825e', metadata={'creationdate': '2024-11-06T10:08:55+00:00', 'keywords': '', 'title': '', 'total_pages': 14, 'page': 0, 'subject': '', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'page_label': '1', 'ptex.fullbanner': 'This i

In [20]:
print(docs[0].page_content)

leverage LangChain’s extensive library of tools, models, and connectors as part


#### Retrievers

A retriever is an interface that returns documents given an unstructured query. It is more general than a vector store. A retriever does not need to be able to store documents, only to return (or retrieve) them. Retrievers can be created from vector stores, but are also broad enough to include other sources.
Retrievers accept a string query as input and return a list of Document objects as output.

Note that all vector stores can be cast to retrievers. Refer to the vector store integration docs for available vector stores. This page lists custom retrievers, implemented via subclassing BaseRetriever.

Define **vectore store backbone retriever and parent document retriever**

1. Vectore Store Backbone Retriever

* Note that the results are identical to the ones obtained using the similarity search strategy.


In [21]:
retriever = doc_search.as_retriever()
docs = retriever.invoke("Langchain")
docs[0]

Document(id='6eb1e3f7-d40c-481f-a04b-c730a590a786', metadata={'author': '', 'moddate': '2024-11-06T10:08:55+00:00', 'source': 'https://www.turing.ac.uk/sites/default/files/2024-11/langchain.pdf', 'subject': '', 'total_pages': 14, 'creator': 'LaTeX with hyperref', 'creationdate': '2024-11-06T10:08:55+00:00', 'title': '', 'producer': 'pdfTeX-1.40.26', 'page': 10, 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'keywords': '', 'page_label': '11'}, page_content='leverage LangChain’s extensive library of tools, models, and connectors as part')


#### What is Parent Document Retriever?

The Parent Document Retriever is a retrieval technique used in Retrieval-Augmented Generation (RAG) systems to balance two important requirements:

1. **Accurate retrieval** using small text chunks.
2. **Rich context** using larger documents.

It stores small chunks for vector search while returning larger parent documents during retrieval.

---

#### The Problem

When documents are split for vector databases, there is a trade-off.

#### Small Chunks

**Advantages**
- More precise embeddings
- Better semantic search accuracy

**Disadvantages**
- Limited context
- Important surrounding information may be lost

#### Large Chunks

**Advantages**
- More context for the language model

**Disadvantages**
- Less accurate embeddings
- Retrieval quality may decrease

---

#### How Parent Document Retriever Solves This

Instead of choosing one approach, it combines both.

#### Parent Document
A larger section of text containing complete context.

#### Child Chunks
Smaller pieces created from the parent document.

The vector database stores embeddings for the child chunks, while the original parent document is stored separately.

---

#### Retrieval Process

#### Step 1: Create Parent Documents

A large document is divided into logical sections.

#### Step 2: Split into Child Chunks

Each parent document is broken into smaller chunks.

#### Step 3: Store Embeddings

Embeddings are generated only for the child chunks and stored in the vector database.

#### Step 4: Perform Similarity Search

When a user submits a query, the system searches the vector database and finds the most relevant child chunk.

#### Step 5: Retrieve Parent Document

Instead of returning only the matching child chunk, the system retrieves the corresponding parent document and sends it to the LLM.

---

#### Benefits

#### Accurate Retrieval
Small chunks produce focused embeddings and improve search quality.

#### Better Context
The LLM receives a larger document section instead of an isolated sentence.

#### Improved Answer Quality
More surrounding information helps the model generate complete and accurate answers.

#### Reduced Context Loss
Important details before and after the matching chunk are preserved.

---

#### Typical Configuration

```text
Parent Chunk Size : 1000–2000 characters
Child Chunk Size  : 200–400 characters

In [22]:
parent_splitter = CharacterTextSplitter(chunk_size = 2000, chunk_overlap=20, separator="\n")
child_splitter = CharacterTextSplitter(chunk_size = 400, chunk_overlap=20, separator='\n')

In [23]:
vectore_store = Chroma(
    collection_name="parent-child-collection", embedding_function=embeddings
)
# The storage layer for the parent documents
doc_store = InMemoryStore()

In [24]:
p_retriever = ParentDocumentRetriever(vectorstore=vectore_store,
                                      docstore=doc_store,
                                      parent_splitter=parent_splitter,
                                      child_splitter=child_splitter)

p_retriever.add_documents(document)

In [25]:
docs = p_retriever.invoke("Langchain")

In [27]:
docs[0].page_content

'maintain optimal performance and quickly resolve issues as they arise.\nLangServe’s streamlined workflow ensures that developers can deploy robust\nAPIs with minimal overhead, making it a practical choice for scaling LLM ap-\nplications in production environments.\n4.3 Integration with LangSmith and LangChain\nLangServe integrates seamlessly with LangSmith, which offers observability fea-\ntures like tracing, logging, and performance monitoring. Through LangSmith,\nLangServe users can track metrics such as request frequency, latency, and er-\nror rates. This integration provides a comprehensive view of API performance,\nenabling developers to optimize applications based on real-time data.\nAdditionally, LangServe’s integration with LangChain allows developers to\nleverage LangChain’s extensive library of tools, models, and connectors as part'

#### Memory

Most LLM applications have a conversational interface. An essential component of a conversation is being able to refer to information introduced earlier in the conversation. At bare minimum, a conversational system should be able to access some window of past messages directly.


##### Chat Message History

One of the core utility classes underpinning most (if not all) memory modules is the `ChatMessageHistory` class. This is a super lightweight wrapper that provides convenience methods for saving `HumanMessages`, `AIMessage`s, and then fetching them all.

Here is an example.


In [29]:
history = ChatMessageHistory()

history.add_ai_message("Hi")
history.add_user_message("What is the longest river in the word")

In [30]:
history.messages

[AIMessage(content='Hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What is the longest river in the word', additional_kwargs={}, response_metadata={})]

In [31]:
ai_response = llm_model.invoke(history.messages)
ai_response

'The Nile River! It stretches for approximately 6,695 kilometers (4,160 miles) from its source in Burundi to its delta on the Mediterranean Sea in Egypt. Would you like to know more about the Nile or another geographical fact?'

#### Conversation Buffer

In [34]:
conversation = ConversationChain(
    llm = llm_model,
    verbose=True,
    memory = ConversationBufferMemory()
)

C:\Users\Vish\AppData\Local\Temp\ipykernel_26096\249880241.py:4: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()
C:\Users\Vish\AppData\Local\Temp\ipykernel_26096\249880241.py:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


In [35]:
conversation.invoke(input="describe honda civic performance")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: describe honda civic performance
AI:

> Finished chain.


{'input': 'describe honda civic performance',
 'history': '',
 'response': "The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda Sensing suite of advanced safety features also plays a significant role in enhancing the overall driving experience. This comprehensive system 

In [36]:
conversation.invoke(input="base on your details i want to buy a one. so which year shouldI buy?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: describe honda civic performance
AI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.

The base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.

In terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed

{'input': 'base on your details i want to buy a one. so which year shouldI buy?',
 'history': "Human: describe honda civic performance\nAI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda Sensing suite of advanced safety features also plays a significant role in enha

In [37]:
conversation.invoke(input="i am also interested with toyota allion 240. what about it?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: describe honda civic performance
AI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.

The base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.

In terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed

{'input': 'i am also interested with toyota allion 240. what about it?',
 'history': "Human: describe honda civic performance\nAI: The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda Sensing suite of advanced safety features also plays a significant role in enhancing the

ConversationChain(memory=ConversationBufferMemory(chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content='describe honda civic performance', additional_kwargs={}, response_metadata={}), AIMessage(content="The Honda Civic! A legendary vehicle with a rich history. The latest model (2022) boasts an impressive performance profile. Let me break it down for you.\n\nThe base model comes with a 2.0-liter naturally aspirated inline-four cylinder engine, producing 158 horsepower and 138 lb-ft of torque. However, if you opt for the higher trim levels or the Sport Touring variant, you'll get the 1.5-liter turbocharged inline-four cylinder engine, which churns out 180 horsepower and 177 lb-ft of torque.\n\nIn terms of acceleration, the Civic can go from 0-60 mph in around 8 seconds, depending on the model and transmission choice. The six-speed manual transmission is available on most trims, while the continuously variable transmission (CVT) comes standard on some models.\n\nThe Honda

### Chains

Chains refer to sequences of calls - whether to an LLM, a tool, or a data preprocessing step.

It combines different LLM calls and actions automatically.

Ex: Summary #1, Summary #2, Summary #3 > Final Summary


In [48]:
template = """Your job is comeup with a classic dish that most popular in given location by the user with a single answer, not descriptively.
                {location}
            your response:
"""

prompt_template = PromptTemplate(template=template, input_variables=['location'])
location_chain = LLMChain(llm=llm_model, prompt=prompt_template, output_key='meal')

location_chain.invoke(input={'location':"china"})

{'location': 'china', 'meal': 'Kung Pao Chicken'}

#### Sequential Chain

In [49]:
template = """Give me the cooking recipe base on the {meal} meal with short and clearly
                your response:
                """

prompt_tem = PromptTemplate(template=template, input_variables=['meal'])
meal_chain = LLMChain(llm=llm_model, prompt=prompt_tem, output_key = 'recipe')


In [50]:
template = """Give me the estimated cooking time base on the recipe given below.
                recipe: {recipe}
                
                your response:
"""

prompt_tem = PromptTemplate(template=template, input_variables=['recipe'])
recipe_chain = LLMChain(llm=llm_model, prompt=prompt_tem, output_key='time')


In [51]:
overall_chain = SequentialChain(chains=[location_chain, meal_chain, recipe_chain],
                                input_variables=['location'],
                                output_variables = ['meal', 'recipe', 'time'],
                                verbose=True)

In [52]:
overall_chain.invoke(input={'location':'Sri Lanka'})



> Entering new SequentialChain chain...

> Finished chain.


{'location': 'Sri Lanka',
 'meal': 'Rice and Curry',
 'recipe': "Here's a simple recipe for a delicious Rice and Curry meal:\n\n**Rice:**\n\n1. Boil 2 cups of water in a pot.\n2. Add 1 cup of uncooked rice, salt to taste.\n3. Cook on medium heat until the water is absorbed.\n\n**Curry:**\n\n1. Heat oil in a pan over medium heat.\n2. Add 1 onion, chopped and sautéed until browned.\n3. Add 2 cloves of garlic, minced and cook for 1 minute.\n4. Add your choice of curry powder or paste (e.g., chicken, beef, or veggie).\n5. Add 1 cup of coconut milk or water.\n6. Simmer for 10-15 minutes until the sauce thickens.\n\n**Serve:**\n\n1. Serve hot rice with the curry on top.",
 'time': "Based on the recipe provided, here are the estimated cooking times:\n\n* **Rice**: approximately 18-20 minutes (boiling water and cooking rice)\n* **Curry**: approximately 15-20 minutes (sauteing onion and garlic, simmering sauce)\n* **Total Cooking Time**: approximately 33-40 minutes\n\nNote: These estimates assu